# Territorial Pulse — EDA, Validation Scientifique et Nowcasting

Analyse exploratoire du panel mensuel (8 provinces de Tanger-Tétouan-Al Hoceïma × 82 mois : 2019-01 à 2025-10), contrôle qualité, corrélations lumière nocturne / HCP, typologie K-Means et évaluation du modèle de nowcasting.

In [ ]:
import json
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

con = duckdb.connect('../data/processed/territorial_pulse.duckdb', read_only=True)
df = con.execute('SELECT * FROM territorial_panel ORDER BY province_id, date').df()
con.close()

with open('../data/processed/model_results.json', encoding='utf-8') as f:
    results = json.load(f)

print(f'Dimensions du panel : {df.shape}')
print('Valeurs manquantes :', df.isna().sum().sum())
df.head()

In [ ]:
# 1. Trajectoires TPI par province (2019-2025)
fig, ax = plt.subplots(figsize=(11, 5))
for pid, grp in df.groupby('province_id'):
    ax.plot(pd.to_datetime(grp['date']), grp['tpi_score'], label=grp['province_name'].iloc[0], linewidth=2)
ax.set_title('Évolution mensuelle du Territorial Pulse Index (TPI) — 2019 à 2025')
ax.set_ylabel('Score TPI (0-100)')
ax.grid(alpha=0.3)
ax.legend(ncol=4, fontsize=8.5)
plt.tight_layout()
plt.show()

In [ ]:
# 2. Validation croisée : Luminosité nocturne VIIRS vs Emploi officiel HCP
fig, ax = plt.subplots(figsize=(8, 5))
for lbl, col in [('Moteur', '#157F5F'), ('En transition', '#E8AA33'), ('Fragile', '#D1495B')]:
    sub = df[df['cluster_label'] == lbl]
    ax.scatter(np.log1p(sub['nightlights_mean']), sub['hcp_employment_index'], alpha=0.55, label=lbl, color=col)
r_val = results['validation']['global_correlations']['nightlights_vs_hcp_pearson']
ax.set_title(f'Luminosité nocturne log(1+VIIRS) vs Emploi HCP (r = {r_val})')
ax.set_xlabel('log(1 + Radiance VIIRS)')
ax.set_ylabel('Indice Emploi HCP')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 3. Backtesting glissant du modèle de nowcasting vs Modèle Naïf
bt = pd.DataFrame(results['nowcasting']['backtest_series'])
print('Métriques Nowcasting :', results['nowcasting']['hcp_nowcast_metrics'])
tanger_bt = bt[bt['province_id'] == 'TTA-01']
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(pd.to_datetime(tanger_bt['date']), tanger_bt['actual_hcp'], label='Emploi HCP réel', linewidth=2)
ax.plot(pd.to_datetime(tanger_bt['date']), tanger_bt['pred_hcp'], label='Nowcast Ridge', linestyle='--', linewidth=2)
ax.plot(pd.to_datetime(tanger_bt['date']), tanger_bt['naive_hcp'], label='Modèle Naïf (t-2)', alpha=0.6)
ax.set_title('Tanger-Assilah — Validation glissante du nowcast (2021-2025)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 4. Sensibilité des poids & Détection d'alertes
print('Sensibilité des poids :', results['tpi_methodology']['sensitivity'])
print('Choc Covid-19 2020 détecté :', results['alerts']['covid_2020_detected'])
pd.DataFrame(results['alerts']['live_alerts'])